# 🧠 Problem 1: Intent-Aware Assistant with Debugging
### LangGraph · MCP-style Tools · Langfuse Tracing · Classification Evaluation

This notebook solves **Problem 1: Intent-Aware Assistant with Debugging**.

It contains:

- ✅ A working intent-aware agent that classifies user input as `info`, `action`, or `summary`
- ✅ An MCP-style client/server tool pattern using JSON-RPC-like `tools/call` requests
- ✅ LangGraph workflow nodes for classification and response generation
- ✅ Langfuse-compatible tracing with a safe local fallback when keys are missing
- ✅ Before vs after classification results
- ✅ Two intentionally wrong classifications in Version 1
- ✅ Accuracy evaluation and fix explanation

> Run all cells top-to-bottom in Google Colab. The notebook runs without an LLM API key. Add Langfuse keys in Colab secrets only when you want hosted traces and screenshots.

---
# 📦 SECTION 1: Install Dependencies

This matches the starter notebook style: install first, then import and configure the environment.

In [ ]:
!pip install -q pandas pydantic langgraph langfuse

print("✅ Packages installed successfully!")

In [1]:
from importlib.metadata import version, PackageNotFoundError

for package_name in ["pandas", "pydantic", "langgraph", "langfuse"]:
    try:
        print(f"{package_name:10s}: {version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name:10s}: not installed")

pandas    : 3.0.3
pydantic  : 2.13.4
langgraph : 1.2.4
langfuse  : 4.7.1


---
# 🔑 SECTION 2: Environment Configuration

Langfuse is optional for local testing. For hosted traces, add these values to Colab secrets or environment variables:

- `LANGFUSE_PUBLIC_KEY`
- `LANGFUSE_SECRET_KEY`
- `LANGFUSE_BASE_URL` or `LANGFUSE_HOST`

The code below does **not** block for input by default, so the notebook can run cleanly in Colab. Set `PROMPT_FOR_KEYS = True` only if you want the cell to ask for keys interactively.

In [2]:
import os
from getpass import getpass

PROMPT_FOR_KEYS = True  # Change to True if you want interactive key entry.


def read_secret(name: str):
    """Read a secret from env vars first, then from Google Colab userdata if available."""
    value = os.environ.get(name)
    if value:
        return value

    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass

    return None


# Langfuse credentials are optional for local verification.
for key in ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL", "LANGFUSE_HOST"]:
    value = read_secret(key)
    if value:
        os.environ[key] = value

# Support both the newer BASE_URL and older HOST names.
if not os.environ.get("LANGFUSE_BASE_URL"):
    os.environ["LANGFUSE_BASE_URL"] = os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com")
if not os.environ.get("LANGFUSE_HOST"):
    os.environ["LANGFUSE_HOST"] = os.environ["LANGFUSE_BASE_URL"]

if PROMPT_FOR_KEYS and not os.environ.get("LANGFUSE_PUBLIC_KEY"):
    os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("LANGFUSE_PUBLIC_KEY: ")
if PROMPT_FOR_KEYS and not os.environ.get("LANGFUSE_SECRET_KEY"):
    os.environ["LANGFUSE_SECRET_KEY"] = getpass("LANGFUSE_SECRET_KEY: ")
if PROMPT_FOR_KEYS and not os.environ.get("LANGFUSE_BASE_URL"):
    os.environ["LANGFUSE_BASE_URL"] = input("LANGFUSE_BASE_URL [https://cloud.langfuse.com]: ").strip() or "https://cloud.langfuse.com"

print("✅ Environment configuration complete")
print("Langfuse host:", os.environ.get("LANGFUSE_BASE_URL"))
print("Langfuse keys present:", bool(os.environ.get("LANGFUSE_PUBLIC_KEY") and os.environ.get("LANGFUSE_SECRET_KEY")))

✅ Environment configuration complete
Langfuse host: https://cloud.langfuse.com
Langfuse keys present: False


---
# 🔭 SECTION 3: Langfuse Tracing Helper

The helper below sends spans to Langfuse when keys are configured. If keys are missing or authentication fails, it still records a local trace table so evaluation remains reproducible.

In [ ]:
import json
import uuid
import datetime as dt
from contextlib import contextmanager
from typing import Any, Dict, Optional

TRACE_EVENTS = []
LANGFUSE_ENABLED = False
langfuse = None

try:
    if os.environ.get("LANGFUSE_PUBLIC_KEY") and os.environ.get("LANGFUSE_SECRET_KEY"):
        from langfuse import get_client
        langfuse = get_client()
        try:
            LANGFUSE_ENABLED = bool(langfuse.auth_check())
        except Exception as auth_error:
            print("⚠️ Langfuse auth_check failed. Using local trace fallback.")
            print("Reason:", auth_error)
            LANGFUSE_ENABLED = False
    else:
        print("ℹ️ Langfuse keys not found. Using local trace fallback.")
except Exception as import_error:
    print("⚠️ Langfuse initialization failed. Using local trace fallback.")
    print("Reason:", import_error)
    LANGFUSE_ENABLED = False


@contextmanager
def trace_span(name: str, input_data: Optional[Dict[str, Any]] = None, metadata: Optional[Dict[str, Any]] = None):
    """Create a Langfuse span when available and always log a local trace event."""
    record = {
        "trace_id": str(uuid.uuid4()),
        "name": name,
        "input": input_data or {},
        "metadata": metadata or {},
        "output": None,
        "start_time": dt.datetime.utcnow().isoformat() + "Z",
        "end_time": None,
        "status": "started",
    }

    if LANGFUSE_ENABLED and langfuse is not None:
        try:
            with langfuse.start_as_current_observation(as_type="span", name=name) as span:
                try:
                    span.update(input=input_data or {})
                except Exception:
                    span.update(input=json.dumps(input_data or {}, default=str))

                yield record

                try:
                    span.update(output=record.get("output"))
                except Exception:
                    span.update(output=json.dumps(record.get("output"), default=str))
        except Exception as trace_error:
            record["status"] = "langfuse_error_fallback"
            record["metadata"]["langfuse_error"] = str(trace_error)
            yield record
    else:
        yield record

    record["end_time"] = dt.datetime.utcnow().isoformat() + "Z"
    if record["status"] == "started":
        record["status"] = "completed"
    TRACE_EVENTS.append(record)


def flush_traces():
    if LANGFUSE_ENABLED and langfuse is not None:
        try:
            langfuse.flush()
            print("✅ Langfuse traces flushed")
        except Exception as flush_error:
            print("⚠️ Langfuse flush failed:", flush_error)
    else:
        print("ℹ️ Local trace mode. No hosted Langfuse flush needed.")

print("✅ Tracing helper ready")
print("Langfuse hosted tracing enabled:", LANGFUSE_ENABLED)

---
# 🧰 SECTION 4: MCP-style Tool Server and Client

This is a lightweight MCP-style pattern:

1. The **server** registers tools.
2. The **client** sends a structured `tools/call` request.
3. The server returns a structured tool response.
4. Every tool call is traced.

The implementation is local and deterministic so it works in Colab without external services.

In [ ]:
from typing import Callable, List


class MCPToolServer:
    """A minimal MCP-style server that exposes tools through JSON-RPC-like calls."""

    def __init__(self, name: str):
        self.name = name
        self._tools: Dict[str, Callable[..., Dict[str, Any]]] = {}

    def register_tool(self, name: str, func: Callable[..., Dict[str, Any]], description: str):
        self._tools[name] = {
            "func": func,
            "description": description,
        }

    def list_tools(self) -> List[Dict[str, str]]:
        return [
            {"name": name, "description": spec["description"]}
            for name, spec in self._tools.items()
        ]

    def handle_request(self, request: Dict[str, Any]) -> Dict[str, Any]:
        if request.get("method") != "tools/call":
            return {"jsonrpc": "2.0", "id": request.get("id"), "error": "Unsupported method"}

        params = request.get("params", {})
        tool_name = params.get("name")
        arguments = params.get("arguments", {})

        if tool_name not in self._tools:
            return {"jsonrpc": "2.0", "id": request.get("id"), "error": f"Unknown tool: {tool_name}"}

        try:
            result = self._tools[tool_name]["func"](**arguments)
            return {"jsonrpc": "2.0", "id": request.get("id"), "result": result}
        except Exception as error:
            return {"jsonrpc": "2.0", "id": request.get("id"), "error": str(error)}


class MCPToolClient:
    """Client that calls tools on the MCPToolServer using structured requests."""

    def __init__(self, server: MCPToolServer):
        self.server = server

    def call_tool(self, name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        request = {
            "jsonrpc": "2.0",
            "id": str(uuid.uuid4()),
            "method": "tools/call",
            "params": {
                "name": name,
                "arguments": arguments,
            },
        }

        with trace_span("mcp_tool_call", input_data=request, metadata={"tool_name": name}) as span:
            response = self.server.handle_request(request)
            span["output"] = response

        if "error" in response:
            raise RuntimeError(response["error"])
        return response["result"]


# ----------------------- Tool Implementations -----------------------
POLICY_KB = {
    "refund": "Refunds are usually processed within 5-7 business days after cancellation is approved.",
    "cancellation": "Cancellation rules depend on the booking type. Flexible bookings can be cancelled before the cutoff time.",
    "documents": "For most support actions, keep booking ID, registered email, payment reference, and travel date ready.",
}


def lookup_policy(topic: str) -> Dict[str, Any]:
    topic_key = topic.lower().strip()
    matched_key = next((key for key in POLICY_KB if key in topic_key), None)
    if not matched_key:
        matched_key = "documents"
    return {
        "tool": "lookup_policy",
        "topic": matched_key,
        "answer": POLICY_KB[matched_key],
    }


def create_support_ticket(issue: str, priority: str = "medium") -> Dict[str, Any]:
    ticket_id = "TKT-" + uuid.uuid4().hex[:8].upper()
    return {
        "tool": "create_support_ticket",
        "ticket_id": ticket_id,
        "priority": priority,
        "status": "created",
        "message": f"Support ticket {ticket_id} created for: {issue}",
    }


def summarize_text(text: str, max_points: int = 3) -> Dict[str, Any]:
    # Simple deterministic summarizer: split into sentences and keep the first few informative ones.
    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
    selected = sentences[:max_points]
    bullets = [f"- {sentence}." for sentence in selected]
    return {
        "tool": "summarize_text",
        "summary": "\n".join(bullets) if bullets else "No meaningful text was provided.",
        "points": len(bullets),
    }


mcp_server = MCPToolServer("IntentAssistantToolServer")
mcp_server.register_tool("lookup_policy", lookup_policy, "Look up static support policy information")
mcp_server.register_tool("create_support_ticket", create_support_ticket, "Create a support ticket for an action request")
mcp_server.register_tool("summarize_text", summarize_text, "Summarize provided text into short bullets")

mcp_client = MCPToolClient(mcp_server)

print("✅ MCP-style server and client ready")
print("Available tools:")
for tool_spec in mcp_server.list_tools():
    print(f" - {tool_spec['name']}: {tool_spec['description']}")

---
# 🧠 SECTION 5: Intent Classifiers

Version 1 is intentionally weak. It checks action keywords too early, which causes two predictable mistakes:

- It treats policy questions containing words like `refund` or `cancelled` as `action`.
- It treats “create a short summary” as `action` even though the real intent is `summary`.

Version 2 fixes the ordering and adds disambiguation logic.

In [ ]:
import re
from typing import Literal, TypedDict

Intent = Literal["info", "action", "summary"]

ACTION_KEYWORDS = [
    "book", "cancel", "create", "raise", "open", "send", "email", "refund", "ticket", "submit", "schedule"
]
SUMMARY_KEYWORDS = ["summarize", "summary", "recap", "tl;dr", "brief"]
INFO_PATTERNS = [
    r"^what\b", r"^how\b", r"^when\b", r"^where\b", r"^why\b",
    r"tell me", r"explain", r"policy", r"rules", r"documents", r"information"
]


def normalize_query(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


def classify_intent_v1(query: str) -> Intent:
    """Buggy classifier: checks action words before summary/info context."""
    q = normalize_query(query)
    if any(keyword in q for keyword in ACTION_KEYWORDS):
        return "action"
    if any(keyword in q for keyword in SUMMARY_KEYWORDS):
        return "summary"
    return "info"


def classify_intent_v2(query: str) -> Intent:
    """Fixed classifier: summary and information questions are disambiguated before action routing."""
    q = normalize_query(query)

    # Explicit summarization intent should win even when the text says "create a summary".
    if any(keyword in q for keyword in SUMMARY_KEYWORDS):
        return "summary"

    # Information-seeking questions should not become actions only because they mention refund/cancel.
    if any(re.search(pattern, q) for pattern in INFO_PATTERNS):
        return "info"

    # Action means the user wants the system to change state or execute an operation.
    if any(keyword in q for keyword in ACTION_KEYWORDS):
        return "action"

    return "info"

print("✅ Intent classifiers created")

---
# 🕸️ SECTION 6: LangGraph Agent Workflow

The graph has two nodes:

`START → classifier → responder → END`

The responder invokes the MCP-style client for all three intent types.

In [ ]:
from langgraph.graph import StateGraph, START, END


class AgentState(TypedDict, total=False):
    user_query: str
    classifier_version: str
    intent: Intent
    response: str
    tool_result: Dict[str, Any]


def classifier_node(state: AgentState) -> Dict[str, Any]:
    query = state["user_query"]
    version = state.get("classifier_version", "v2")

    with trace_span("classify_intent", input_data={"query": query, "version": version}) as span:
        if version == "v1":
            intent = classify_intent_v1(query)
        else:
            intent = classify_intent_v2(query)
        span["output"] = {"intent": intent}

    print(f"[Classifier {version}] Intent: {intent}")
    return {"intent": intent}


def responder_node(state: AgentState) -> Dict[str, Any]:
    query = state["user_query"]
    intent = state["intent"]

    with trace_span("respond_by_intent", input_data={"query": query, "intent": intent}) as span:
        if intent == "info":
            tool_result = mcp_client.call_tool("lookup_policy", {"topic": query})
            response = f"Information Answer:\n{tool_result['answer']}"

        elif intent == "action":
            priority = "high" if any(word in normalize_query(query) for word in ["urgent", "failed", "not credited"]) else "medium"
            tool_result = mcp_client.call_tool("create_support_ticket", {"issue": query, "priority": priority})
            response = f"Action Completed:\n{tool_result['message']}\nStatus: {tool_result['status']}"

        elif intent == "summary":
            # Use the text after ':' when present; otherwise summarize the full query.
            text_to_summarize = query.split(":", 1)[1].strip() if ":" in query else query
            tool_result = mcp_client.call_tool("summarize_text", {"text": text_to_summarize, "max_points": 3})
            response = f"Structured Summary:\n{tool_result['summary']}"

        else:
            tool_result = {"tool": None, "error": "Unknown intent"}
            response = "I could not determine the right action for this request."

        span["output"] = {"response": response, "tool_result": tool_result}

    print(f"[Responder] Used tool: {tool_result.get('tool')}")
    return {"response": response, "tool_result": tool_result}


workflow = StateGraph(AgentState)
workflow.add_node("classifier", classifier_node)
workflow.add_node("responder", responder_node)
workflow.add_edge(START, "classifier")
workflow.add_edge("classifier", "responder")
workflow.add_edge("responder", END)

app = workflow.compile()

print("✅ LangGraph agent compiled")
print("Graph flow: START → classifier → responder → END")

In [ ]:
# Optional graph visualization. If graph image rendering is unavailable, a text flow is printed.
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("Graph visualization not available here.")
    print("START → classifier → responder → END")

---
# 🚀 SECTION 7: Run the Working Agent

The helper below runs the graph and prints the final response.

In [ ]:
from IPython.display import display
import pandas as pd


def run_agent(query: str, classifier_version: str = "v2") -> Dict[str, Any]:
    print("=" * 80)
    print(f"🙋 User: {query}")
    print(f"Classifier version: {classifier_version}")
    print("-" * 80)

    with trace_span("agent_run", input_data={"query": query, "classifier_version": classifier_version}) as span:
        result = app.invoke({"user_query": query, "classifier_version": classifier_version})
        span["output"] = result

    print("\n🤖 Agent Response:")
    print(result["response"])
    print("=" * 80)
    return result


# Demo examples for each intent.
demo_queries = [
    "What is the refund policy for a cancelled ticket?",
    "Open a support ticket because my refund was not credited.",
    "Summarize these notes: The customer missed the bus. Payment was successful. Agent should check refund eligibility.",
]

for demo_query in demo_queries:
    _ = run_agent(demo_query, classifier_version="v2")

---
# 📊 SECTION 8: Evaluation Dataset

The dataset is deliberately small and easy to inspect. Version 1 should have exactly two wrong classifications.

In [ ]:
TEST_CASES = [
    {
        "query": "What is the refund policy for a cancelled ticket?",
        "expected_intent": "info",
        "reason": "The user asks for policy information, not a refund action.",
    },
    {
        "query": "Create a short summary from these notes: Customer paid twice. First payment failed visually, second payment succeeded.",
        "expected_intent": "summary",
        "reason": "The requested artifact is a summary, not a system action.",
    },
    {
        "query": "Summarize these meeting notes: Refund queue is high. Add one support agent. Review pending tickets daily.",
        "expected_intent": "summary",
        "reason": "Explicit summarization request.",
    },
    {
        "query": "Open a support ticket for payment failure on booking RB123.",
        "expected_intent": "action",
        "reason": "The user asks the assistant to create/open a ticket.",
    },
    {
        "query": "Tell me what documents are needed for support verification.",
        "expected_intent": "info",
        "reason": "The user asks for information.",
    },
    {
        "query": "Send an email update to the customer about ticket TKT-100.",
        "expected_intent": "action",
        "reason": "The user asks the assistant to send an update.",
    },
]

pd.DataFrame(TEST_CASES)

In [ ]:
def evaluate_classifier(classifier_version: str) -> pd.DataFrame:
    rows = []
    classifier = classify_intent_v1 if classifier_version == "v1" else classify_intent_v2

    for item in TEST_CASES:
        query = item["query"]
        expected = item["expected_intent"]
        with trace_span(
            "classification_evaluation",
            input_data={"query": query, "expected": expected, "version": classifier_version},
        ) as span:
            predicted = classifier(query)
            correct = predicted == expected
            row = {
                "version": classifier_version,
                "query": query,
                "expected": expected,
                "predicted": predicted,
                "correct": correct,
                "reason": item["reason"],
            }
            span["output"] = row
            rows.append(row)

    return pd.DataFrame(rows)


v1_eval = evaluate_classifier("v1")
v2_eval = evaluate_classifier("v2")

all_eval = pd.concat([v1_eval, v2_eval], ignore_index=True)
accuracy_df = (
    all_eval.groupby("version")["correct"]
    .mean()
    .reset_index(name="classification_accuracy")
)
accuracy_df["classification_accuracy"] = (accuracy_df["classification_accuracy"] * 100).round(2)

print("Classification Accuracy")
display(accuracy_df)

print("Detailed Evaluation")
display(all_eval)

---
# 🐞 SECTION 9: Identify 2 Wrong Classifications

The next cell isolates the incorrect Version 1 outputs.

In [ ]:
wrong_v1 = v1_eval[v1_eval["correct"] == False].copy()
print(f"Version 1 wrong classifications found: {len(wrong_v1)}")
display(wrong_v1[["query", "expected", "predicted", "reason"]])

---
# 🔧 SECTION 10: Before vs After Output

This cell runs the same incorrectly classified Version 1 cases through both classifiers and shows the response difference.

In [ ]:
before_after_rows = []

for query in wrong_v1["query"].tolist():
    before = app.invoke({"user_query": query, "classifier_version": "v1"})
    after = app.invoke({"user_query": query, "classifier_version": "v2"})

    before_after_rows.append({
        "query": query,
        "v1_intent_before_fix": before["intent"],
        "v1_tool_before_fix": before["tool_result"].get("tool"),
        "v1_response_before_fix": before["response"],
        "v2_intent_after_fix": after["intent"],
        "v2_tool_after_fix": after["tool_result"].get("tool"),
        "v2_response_after_fix": after["response"],
    })

before_after_df = pd.DataFrame(before_after_rows)
display(before_after_df)

---
# 📝 SECTION 11: Fix Explanation

In [ ]:
fix_explanation = """
Fix Explanation
===============

Problem in Version 1:
- The classifier checked action keywords first.
- Words such as "refund", "cancelled", and "create" triggered ACTION even when the user was asking for information or asking to create a summary.

Changes in Version 2:
1. Explicit summary signals are checked first: summarize, summary, recap, tl;dr, brief.
2. Information-seeking patterns are checked before action routing: what/how/when/tell me/explain/policy/rules/documents.
3. Action intent is used only after summary and info patterns are ruled out.

Result:
- The policy question is now classified as INFO.
- The "create a short summary" request is now classified as SUMMARY.
- Accuracy improves from the Version 1 score to the Version 2 score shown above.
"""

print(fix_explanation)

---
# 🔭 SECTION 12: Trace Review and Screenshot Checklist

If Langfuse keys are configured, open the Langfuse dashboard and capture screenshots for these trace names:

- `agent_run`
- `classify_intent`
- `respond_by_intent`
- `mcp_tool_call`
- `classification_evaluation`

For local verification, the trace table below shows the same trace events captured in memory.

In [ ]:
trace_df = pd.DataFrame(TRACE_EVENTS)
print(f"Total trace events captured locally: {len(trace_df)}")
if len(trace_df) > 0:
    display(trace_df[["name", "status", "start_time", "end_time", "metadata"]].tail(20))

flush_traces()
print("\n✅ Problem 1 notebook complete")
print("If hosted tracing is enabled, take screenshots from:", os.environ.get("LANGFUSE_BASE_URL", "https://cloud.langfuse.com"))